**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/hiraoka-2025"], check=True)
    os.chdir("/content/encore/entries/hiraoka-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Hiraoka & Jo (2025) on DANDI:000044: the other neurons, exponent capped at 3 (a variant)

Paper: T. Hiraoka and H.-H. Jo, "Hierarchical organization of bursty trains in event
sequences", doi:10.1063/5.0296506, read from the arXiv preprint body under `paper/`.
Dataset: DANDI:000044 version 0.210812.1516, the version the paper cites (ref. 39); see the
header of `reproduce_hiraoka_2025.ipynb` for the choice of version and the claims table.
This notebook is a variant of `other_neurons.ipynb` for `fig2-other-neurons-similar`: the same
test with the default range of the exponent of powerlaw 2.0.0, [0, 3], which the adopted fit
does not use (helper, `READINGS`). It runs on grid (a) alone, for run time (about 11 minutes on
4 processes), and counts the fits at the bound.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1"); os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import powerlaw
import bursttree as bt

T0 = time.time()
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 300)
print("dandiset", bt.DANDISET, "version", bt.DANDI_VERSION)
print("powerlaw", powerlaw.__version__, "; capped readings:", bt.CAPPED_READINGS)

## Data check

The units tables of all eight sessions: counts by cell type, and spike trains that are not
sorted in time (the burst analysis sorts each train before taking IETs).

In [ ]:
assets = bt.list_assets()
trains, rows = [], []
for a in assets:
    ses = a["path"].split("ses-")[1].split("_")[0]
    U = bt.load_units(bt.open_nwb(a["url"]))
    uns = [int(U["id"][i]) for i, s in enumerate(U["spikes"]) if np.any(np.diff(s) < 0)]
    rows.append(dict(session=ses, units=len(U["id"]), excitatory=int((U["cell_type"] == "excitatory").sum()),
                     inhibitory=int((U["cell_type"] == "inhibitory").sum()), unsorted_unit_ids=uns))
    for i, s in enumerate(U["spikes"]):
        trains.append(dict(session=ses, row=i, id=int(U["id"][i]), cell_type=U["cell_type"][i], n=len(s),
                           unsorted=int(U["id"][i]) in uns, spikes_ms=s * 1e3))
print(pd.DataFrame(rows).to_string())
print("units:", len(trains), "; by cell type:", pd.Series([t["cell_type"] for t in trains]).value_counts().to_dict(),
      "; with unsorted spike times:", [(t["session"], t["id"], t["cell_type"]) for t in trains if t["unsorted"]])
for ct in ("excitatory", "inhibitory"):
    ns = [t["n"] for t in trains if t["cell_type"] == ct]
    print(f"spikes per {ct} unit: min", min(ns), "median", int(np.median(ns)), "max", max(ns))

## fig2-other-neurons-similar, exponent capped at 3

The criterion as in `other_neurons.ipynb`, on grid (a), with the sets of that notebook.

In [ ]:
def summarise(d):
    used = d[d.used]
    return pd.Series(dict(units=len(d), used=len(used), pattern=int(used.pattern.sum()),
                          frac_pattern=round(used.pattern.mean(), 3) if len(used) else np.nan,
                          median_b_frac=round(used.b_frac.median(), 3), median_k_frac=round(used.k_frac.median(), 3),
                          beta_q25=round(used.median_beta.quantile(0.25), 3), beta_median=round(used.median_beta.median(), 3),
                          beta_q75=round(used.median_beta.quantile(0.75), 3)))

SETS = {"(a) pyramidal, Achilles-10252013": lambda d: d[(d.session == "Achilles-10252013") & (d.cell_type == "excitatory")],
        "(b) pyramidal, all sessions": lambda d: d[d.cell_type == "excitatory"],
        "variant: inhibitory, Achilles-10252013": lambda d: d[(d.session == "Achilles-10252013") & (d.cell_type == "inhibitory")],
        "variant: inhibitory, all sessions": lambda d: d[d.cell_type == "inhibitory"],
        "variant: all units, Achilles-10252013": lambda d: d[d.session == "Achilles-10252013"],
        "variant: all units, all sessions": lambda d: d}
t1 = time.time()
with ProcessPoolExecutor(max_workers=4) as ex:
    resc = list(ex.map(bt.capped_patterns, [t["spikes_ms"] for t in trains], chunksize=2))
print(f"{len(trains)} units fitted (capped exponent, grid a) in {(time.time() - t1) / 60:.1f} min")
RC = pd.DataFrame([dict(session=t["session"], id=t["id"], cell_type=t["cell_type"], n_spikes=t["n"], unsorted=t["unsorted"], **r)
                   for t, rr in zip(trains, resc) for r in rr])
for name, sel in SETS.items():
    d = sel(RC)
    s_ = d.groupby("reading").apply(summarise)[["used", "pattern", "frac_pattern", "beta_median"]]
    u_ = d[d.used].groupby("reading")[["n_at_cap", "n_fits"]].sum()
    print(f"\nset {name}, capped, grid (a):"); print(s_.join(u_).to_string())
print(f"\nnotebook time {(time.time() - T0) / 60:.1f} min, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (main process)")

**Result (variant).** With the exponent capped at 3, on grid (a), the fraction of pyramidal units
showing the pattern in set (b) is 0.547 under the continuous KS fit, 0.233 under the discrete KS
fit, 0.906 under the continuous fit from xmin = 1 and 0.211 under the discrete one; in set (a)
0.400, 0.117, 0.892 and 0.017. The cap lifts the continuous KS fit above one half in set (b)
(0.547, against 0.449 unbounded); that fit sits at the bound in 1034 of the 9576 fits of set (b),
and the discrete KS fit in 78. The outcome does not depend on the cap: the continuous fit from
xmin = 1 passes under both ranges.